# Desafio #22 — Degradação de bateria: alerta ANTES do limiar (PoC)

PoC de **diagnóstico preditivo de bateria** para o desafio *Powertrain Elétrico*.
Demonstra: **índice de saúde (SoH) caindo → alerta disparando antes do fim de vida → lead time**.

**Dataset:** NASA PCoE Li-ion (B0005/6/7/18) — arranque rápido e curva de degradação limpa.

> **Enquadramento honesto:** não há DTC em dado público. Aqui o "evento" é o cruzamento do
> **limiar de degradação (SoH)**. Reporta-se *alerta antes do limiar*, **não** *antes do DTC*.
> Severson/EVBattery entram depois como escala/EV real; o dado do patrocinador traz o DTC real.


## 1. Setup — clonar o repositório


In [ ]:
import os, sys
REPO = 'ev-battery-early-degradation-poc'
URL  = 'https://github.com/GuilhermeFrick/' + REPO + '.git'
if not os.path.isdir(REPO):
    !git clone --depth 1 $URL
%cd {REPO}
!git pull --no-edit 2>/dev/null || true
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd =', os.getcwd())


In [ ]:
# scipy/numpy/matplotlib já vêm no Colab; garante versões
!pip -q install 'scipy>=1.10' 2>/dev/null
import numpy as np, matplotlib.pyplot as plt
print('ok')


## 2. Dados NASA (.mat)

Coloque os arquivos **B0005.mat, B0006.mat, B0007.mat, B0018.mat** em `data/`.
Baixe do NASA PCoE Prognostics Data Repository (Li-ion Battery). Se tiver os arquivos,
use o upload abaixo; ou defina `URL_MAT` para um arquivo direto.


In [ ]:
import os; os.makedirs('data', exist_ok=True)
# Opção A — upload manual (selecione os .mat):
try:
    from google.colab import files
    up = files.upload()
    for name in list(up):
        os.replace(name, os.path.join('data', name))
except Exception as e:
    print('upload manual indisponível fora do Colab:', type(e).__name__)
# Opção B — URL direto (descomente e ajuste):
# URL_MAT = 'https://.../B0005.mat'
# !wget -q -P data "$URL_MAT"
!echo 'arquivos em data/:' ; ls -la data/*.mat 2>/dev/null || echo '(nenhum .mat ainda)'


## 3. Rodar a PoC — SoH, alerta antes do limiar, lead time


In [ ]:
import json
from battery_poc import load_nasa_dir, analyze_fleet, summarize, plot_fleet, plot_lead_times

caps = load_nasa_dir('data')                 # capacidade de descarga por ciclo
print('células:', {c: len(v) for c, v in caps.items()})

results = analyze_fleet(caps, eol_soh=0.80, window=20, rul_warn=50)
for c, r in results.items():
    print(f'  {c}: EOL={r.eol_idx}  alerta={r.alert_idx}  lead={r.lead_time}  falso_alarme={r.false_alarm}')

summ = summarize(results)
print('\nRESUMO:', json.dumps(summ, ensure_ascii=False))


## 4. Figuras — a prova visual


In [ ]:
os.makedirs('docs/figuras', exist_ok=True)
plot_fleet(results, 'docs/figuras/soh-alerta.png'); plt.show()


In [ ]:
plot_lead_times(results, 'docs/figuras/lead-time.png'); plt.show()


## 5. Leitura e próximos passos

- **Índice de saúde (SoH) cai** e o **alerta dispara antes** do cruzamento do limiar de fim de vida
  — com o **lead time** (ciclos de antecedência) medido por célula. É a prova do método.
- **Honestidade:** poucas células → isto é *estudo de degradação + alerta por tendência*, não um
  preditor treinado que generaliza para frota. Vale como PoC e como base para escalar.

**Escala (mesma estrutura):**
1. **Severson/MIT** — previsão precoce de vida a partir dos primeiros ~100 ciclos (mais células).
2. **EVBattery** — anomalia em **EV real** (label de anomalia/capacidade).
3. **Dado do patrocinador** — traz o **DTC real** → aí sim mede-se *lead time antes do DTC* e
   vira preditivo supervisionado. Código em pandas/numpy, portável para **Databricks** (I/O isolado).
